# 04 — Conversation History

This notebook covers the **history** item from the project roadmap (see README): extending notebook 03's single-shot `ask()` into a multi-turn conversation. Follow-up questions from the operator often depend on what was already asked/answered — this notebook demonstrates threading `chat_history` through both retrieval (via history-aware query reformulation) and generation.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from factory_floor.config import VECTOR_DIR, COLLECTION_NAME

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY in a .env file before running this notebook.')


## 1. Load the existing vector database

No PDF ingestion or embedding should happen here. That work was already completed in notebook 02 and cached in Chroma.

In [2]:
from factory_floor.vectorstore import get_embeddings, load_vectorstore
from factory_floor.rag import build_retriever

embeddings = get_embeddings()
vectorstore = load_vectorstore(VECTOR_DIR, COLLECTION_NAME, embeddings=embeddings)
retriever = build_retriever(vectorstore, k=5)
print('Persistent retriever loaded.')


Persistent retriever loaded.


## 2. Why follow-up questions need history-aware retrieval

A follow-up like *"what if that fault code has already cleared?"* means nothing to a vector search on its own — the embedding of "that fault code" carries no equipment-specific signal. `factory_floor.rag.contextualize_question` rewrites the follow-up into a standalone question (using the conversation so far) **before** retrieval runs, so the search actually has something specific to match against. This only costs an extra LLM call on turns after the first — the very first question in a conversation is never rewritten.

In [3]:
from factory_floor.rag import ask, build_chat_history, get_llm

llm = get_llm()
turns = []


## 3. Turn 1 — initial question

In [4]:
question_1 = 'A SINAMICS G120 drive is repeatedly tripping. What should a technician inspect before deciding on a cause?'

turns.append(ask(question_1, retriever, llm, chat_history=build_chat_history(turns)))

print(turns[-1]['answer'])
print('\nSOURCES USED')
print(turns[-1]['sources'])


For a SINAMICS G120 drive that is repeatedly tripping, a technician should inspect the following before deciding on a cause:

1. **Check for DC link overvoltage conditions**:
   - Verify if the power unit has detected an overvoltage in the DC link, which can be caused by motor regenerating too much energy, line supply voltage too high, line phase interruption, DC link voltage control switched off, or improper dynamic response of the DC link voltage controller.
   - Check the DC link voltage at the time of trip (parameter r0949).
   - Remedies include increasing ramp-down time (p1121), setting rounding times (p1130, p1136), and activating the DC link voltage controller (p1240, p1280) [SOURCE 1].

2. **Inspect line supply and infeed**:
   - Check the line supply voltage at the input terminals and verify the line supply voltage setting (p0210).
   - Check the infeed unit and its environment including line supply, filters, reactors, and fuses.
   - Ensure precharging resistors have cooled 

## 4. Turn 2 — an elliptical follow-up

The next question only makes sense in light of turn 1 — it never repeats "SINAMICS G120" or "tripping". We print the rewritten `standalone_question` so the reformulation step is visible, not just trusted.

In [5]:
question_2 = 'What if that fault code has already cleared by the time I check?'

turns.append(ask(question_2, retriever, llm, chat_history=build_chat_history(turns)))

print('STANDALONE QUESTION (rewritten by the LLM):')
print(turns[-1]['standalone_question'])
print('\nANSWER:')
print(turns[-1]['answer'])
print('\nSOURCES USED')
print(turns[-1]['sources'])


STANDALONE QUESTION (rewritten by the LLM):
If the fault code related to the SINAMICS G120 drive tripping has already cleared by the time you check, what diagnostic steps or data should a technician review to determine the cause of the transient fault?

ANSWER:
If the fault code has already cleared by the time you check, the documentation does not provide a direct procedure for post-clearance analysis. However, you should:

- Review the drive's fault history or diagnostic logs if available to identify the exact fault code and conditions at the time of the trip.
- Check parameters related to the fault type that was indicated before clearing, such as line supply voltage (p0210), DC link voltage, or motor data identification status (p1900) depending on the fault.
- Inspect the physical wiring and connections, especially DRIVE-CLiQ wiring for communication faults, and the infeed unit and environment for line faults or supply issues.
- Verify that the motor data identification has been perf

## 5. Turn 3 — a second follow-up

One more hop, to confirm the conversation keeps working beyond a single follow-up.

In [6]:
question_3 = 'And which of those checks needs the drive powered down first?'

turns.append(ask(question_3, retriever, llm, chat_history=build_chat_history(turns)))

print('STANDALONE QUESTION (rewritten by the LLM):')
print(turns[-1]['standalone_question'])
print('\nANSWER:')
print(turns[-1]['answer'])
print('\nSOURCES USED')
print(turns[-1]['sources'])


STANDALONE QUESTION (rewritten by the LLM):
Which of the inspections and checks related to diagnosing repeated tripping or cleared fault codes on a SINAMICS G120 drive require the drive to be powered down before performing them?

ANSWER:
From the retrieved documentation, the checks that require the drive to be powered down first include:

- **Disconnecting the infeed unit from the line supply** to allow precharging resistors to cool down safely before further inspection or work on the infeed or line supply components [SOURCE 2].

- **Checking or replacing hardware components** such as the Control Unit, Motor Module, Power Module, or Sensor Module, which triggers the need for a partial acceptance test and cannot be done while the drive is powered [SOURCE 4].

- **Inspecting or repairing DRIVE-CLiQ wiring** and connections, since internal communication faults require checking physical wiring and EMC-compliant installation, which typically involves powering down the drive for safe handlin

## Milestone checkpoint

The pipeline is now:

`follow-up question → history-aware reformulation → retrieval → contextualized answer`

This is exactly what the Streamlit app's follow-up box (rendered below each answer) exposes to the operator. Per the README roadmap, **history** is done — vision, agents, memory and safety validation remain.